# SAM 3D Body — CPU Offload

A modified version of Meta's [SAM 3D Body](https://github.com/facebookresearch/sam-3d-body) with **CPU/RAM offloading** and an **extended biomechanics suite**.

**Memory profile:** All model weights live in CPU RAM at idle (~3 MB allocated VRAM). During inference, weights hop to GPU (~6.5 GiB peak) then immediately return to CPU RAM.

**Biomechanics computed:** Lower body (knee/hip/ankle flexion, valgus, KWR), upper body (elbow, shoulder rotation), posture (trunk lean, forward lean, head tilt), risk scores (Dynamic Valgus Score, LESS proxy, asymmetry indices), and temporal metrics (angular velocity, CoM sway/acceleration, cadence, range of motion).

**Hardware tested:** Kaggle dual T4 (2× 16 GiB VRAM), CUDA 12.6, Python 3.12.

## 1. Installation

In [ ]:
!pip install --quiet timm einops pycocotools huggingface_hub opencv-python triton psutil notebook boxmot braceexpand matplotlib roma omegaconf pytorch_lightning yacs pyrender fvcore black iopath==0.1.7 cloudpickle hydra-core tensorboard

In [ ]:
!pip install --quiet 'git+https://github.com/facebookresearch/detectron2.git@a1ce2f9' --no-build-isolation --no-deps

In [ ]:
!pip install --quiet git+https://github.com/microsoft/MoGe.git

In [ ]:
!pip install -qU gradio

## 2. Load patched source

The patched SAM 3D Body source (with CPU offloading) is stored as a Kaggle dataset and copied into the working directory.

In [ ]:
!cp -r /kaggle/input/datasets/aminefezzani/sam3d-body/sam3d_offloaded/* ./

## 3. HuggingFace authentication

Weights are streamed directly from `facebook/sam-3d-body-dinov3`. A HF read token with access to the gated model is required.

In [ ]:
from kaggle_secrets import UserSecretsClient
hf_token = UserSecretsClient().get_secret("hf_token")

from huggingface_hub import login
login(token=hf_token)

## 4. Imports

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from notebook.utils import setup_sam_3d_body
from tools.vis_utils import visualize_sample_together, visualize_sample
from sam_3d_body.visualization.renderer import Renderer
from sam_3d_body.visualization.skeleton_visualizer import SkeletonVisualizer
from sam_3d_body.metadata.mhr70 import pose_info as mhr70_pose_info
from boxmot.trackers.bytetrack.bytetrack import ByteTrack

## 5. Load models

`setup_sam_3d_body` initialises all four models with `offload_mode="cpu"` by default:

| Model | Purpose | Size |
|---|---|---|
| SAM 3D Body (ViT-H + MHR70) | 3D body mesh reconstruction | ~3 GB |
| ViTDet | Human detection | ~1.5 GB |
| MoGe2 | Monocular FOV estimation | ~1 GB |

All weights land in **CPU RAM**. Nothing is pinned to VRAM until inference starts.

In [ ]:
LIGHT_BLUE = (0.65098039, 0.74117647, 0.85882353)

skeleton_visualizer = SkeletonVisualizer(line_width=2, radius=5)
skeleton_visualizer.set_pose_meta(mhr70_pose_info)

In [ ]:
# Set up the estimator
estimator = setup_sam_3d_body(hf_repo_id="facebook/sam-3d-body-dinov3")

### Verify idle VRAM

After loading, ~3 MB should be allocated. The ~1.4 GiB reserved is PyTorch's caching allocator — not a leak.

In [ ]:
import torch
print(f"Allocated : {torch.cuda.memory_allocated()/1024**2:.1f} MB")
print(f"Reserved  : {torch.cuda.memory_reserved()/1024**2:.1f} MB")
print("Expected  : ~3 MB allocated, ~1400 MB reserved")

## 6. Biomechanics

All metrics computed from 3D keypoints (`pred_keypoints_3d`) with 2D fallback.

### Single-frame metrics

| Category | Metrics |
|---|---|
| **Lower body** | Knee/hip/ankle flexion (L+R), knee valgus (L+R), knee-width ratio, stride length proxy, ground contact foot |
| **Posture** | Trunk lean, full-body forward lean, head tilt, hip drop, shoulder rotation |
| **Upper body** | Elbow flexion (L+R) |
| **Risk** | Dynamic Valgus Score (0–10), LESS flag, asymmetry index for all bilateral joints |

### Temporal metrics (video only)

| Metric | Description |
|---|---|
| Angular velocity | Knee + hip deg/s — spikes indicate explosive movements |
| CoM sway / accel | Lateral centre-of-mass displacement and acceleration |
| Cadence | Foot-strike events per second |
| Range of motion | Max−min joint angle over a rolling 15-frame window |

### Colour coding on HUD
🟢 Normal range · 🟠 Mild deviation · 🔴 Concerning

In [ ]:
# ── MHR70 joint indices ──────────────────────────────────────────────────────
J = {
    "pelvis":       0,
    "l_hip":        1,   "r_hip":        2,
    "spine1":       3,
    "l_knee":       4,   "r_knee":        5,
    "spine2":       6,
    "l_ankle":      7,   "r_ankle":       8,
    "spine3":       9,
    "l_foot":      10,   "r_foot":       11,
    "neck":        12,
    "l_collar":    13,   "r_collar":     14,
    "head":        15,
    "l_shoulder":  16,   "r_shoulder":   17,
    "l_elbow":     18,   "r_elbow":      19,
    "l_wrist":     20,   "r_wrist":      21,
}


# ── Geometry helpers ──────────────────────────────────────────────────────────

def angle_between(a, b, c):
    """Angle at joint b formed by segments b→a and b→c, in degrees."""
    v1 = a - b; v2 = c - b
    cos_a = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1, 1))))

def valgus_angle(hip, knee, ankle):
    """Medial knee deviation from hip-ankle line in the frontal plane (XY)."""
    ref2 = (ankle - hip)[:2]; dev2 = (knee - hip)[:2]
    if np.linalg.norm(ref2) < 1e-6 or np.linalg.norm(dev2) < 1e-6:
        return 0.0
    cos_a = np.dot(ref2, dev2) / (np.linalg.norm(ref2) * np.linalg.norm(dev2) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1, 1))))

def vec_angle_from_vertical(vec, use_3d=True):
    """Angle of a vector from the vertical Y axis, in degrees."""
    vertical = np.array([0., 1., 0.]) if use_3d else np.array([0., 1.])
    v = vec[:3] if use_3d else vec[:2]
    cos_a = np.dot(v, vertical) / (np.linalg.norm(v) + 1e-8)
    return float(np.degrees(np.arccos(np.clip(cos_a, -1, 1))))

def asymmetry_index(left, right):
    """Standard asymmetry index as a percentage."""
    mean = (abs(left) + abs(right)) / 2
    return float(abs(left - right) / (mean + 1e-8) * 100)


# ── Temporal buffer ───────────────────────────────────────────────────────────

class TemporalBuffer:
    """
    Maintains a rolling window of per-frame metric dicts.
    Pass the same instance across all frames of a video.
    For single-image use, pass None to compute_biomechanics.
    """
    def __init__(self, fps=30.0, window=10):
        self.fps    = fps
        self.window = window
        self.frames = []   # list of metric dicts

    def push(self, metrics):
        self.frames.append(metrics)
        if len(self.frames) > self.window:
            self.frames.pop(0)

    def last(self, key, n=2):
        vals = [f[key] for f in self.frames[-n:] if key in f]
        return vals

    def delta(self, key):
        vals = self.last(key, 2)
        return (vals[-1] - vals[0]) if len(vals) == 2 else 0.0

    def range_of_motion(self, key):
        vals = [f[key] for f in self.frames if key in f]
        return (max(vals) - min(vals)) if vals else 0.0


# ── Main biomechanics function ────────────────────────────────────────────────

def compute_biomechanics(person_output, buf=None, fps=30.0):
    """
    Compute all biomechanical metrics for a single person.

    Args:
        person_output : dict from estimator.process_one_image()
        buf           : TemporalBuffer instance (None for single-image)
        fps           : video frame rate, used for velocity calculations

    Returns a flat dict with all metrics.
    """
    kp      = person_output.get("pred_keypoints_3d", person_output.get("pred_keypoints_2d"))
    use_3d  = "pred_keypoints_3d" in person_output
    verts   = person_output.get("pred_vertices", None)

    def pt(name):
        return kp[J[name]]

    # ── Centre of mass ────────────────────────────────────────────────────────
    com = np.mean(verts, axis=0) if verts is not None else pt("pelvis")

    # ── Lower body ───────────────────────────────────────────────────────────
    l_knee_deg = angle_between(pt("l_hip"),  pt("l_knee"),  pt("l_ankle"))
    r_knee_deg = angle_between(pt("r_hip"),  pt("r_knee"),  pt("r_ankle"))
    l_hip_deg  = angle_between(pt("spine1"), pt("l_hip"),   pt("l_knee"))
    r_hip_deg  = angle_between(pt("spine1"), pt("r_hip"),   pt("r_knee"))
    l_ankle_deg = angle_between(pt("l_knee"), pt("l_ankle"), pt("l_foot"))
    r_ankle_deg = angle_between(pt("r_knee"), pt("r_ankle"), pt("r_foot"))
    l_val = valgus_angle(pt("l_hip"), pt("l_knee"), pt("l_ankle"))
    r_val = valgus_angle(pt("r_hip"), pt("r_knee"), pt("r_ankle"))

    knee_w = np.linalg.norm(pt("l_knee")[:2] - pt("r_knee")[:2])
    hip_w  = np.linalg.norm(pt("l_hip")[:2]  - pt("r_hip")[:2])
    kwr    = float(knee_w / (hip_w + 1e-8))
    hip_drop = float((pt("l_hip")[1] - pt("r_hip")[1]) * 100)

    # Stride: horizontal ankle separation (proxy for stride length)
    stride_len = float(np.linalg.norm((pt("l_ankle") - pt("r_ankle"))[:2]))

    # Ground contact: foot closest to ground (min Y)
    l_foot_h = float(pt("l_foot")[1])
    r_foot_h = float(pt("r_foot")[1])
    ground_contact = "left" if l_foot_h <= r_foot_h else "right"

    # ── Trunk / posture ──────────────────────────────────────────────────────
    trunk_vec   = pt("neck") - pt("pelvis")
    trunk_lean  = vec_angle_from_vertical(trunk_vec, use_3d)

    body_vec    = pt("head") - pt("l_ankle") * 0.5 - pt("r_ankle") * 0.5
    forward_lean = vec_angle_from_vertical(body_vec, use_3d)

    head_vec    = pt("head") - pt("neck")
    head_tilt   = vec_angle_from_vertical(head_vec, use_3d)

    # ── Upper body ───────────────────────────────────────────────────────────
    l_elbow_deg = angle_between(pt("l_shoulder"), pt("l_elbow"), pt("l_wrist"))
    r_elbow_deg = angle_between(pt("r_shoulder"), pt("r_elbow"), pt("r_wrist"))

    # Shoulder rotation: angle of shoulder line vs hip line in the transverse plane
    shoulder_vec = (pt("l_shoulder") - pt("r_shoulder"))[:2]
    hip_vec_2d   = (pt("l_hip")      - pt("r_hip"))[:2]
    cos_sh = np.dot(shoulder_vec, hip_vec_2d) / (
        np.linalg.norm(shoulder_vec) * np.linalg.norm(hip_vec_2d) + 1e-8)
    shoulder_rot = float(np.degrees(np.arccos(np.clip(cos_sh, -1, 1))))

    # ── Asymmetry indices ────────────────────────────────────────────────────
    asym_knee   = asymmetry_index(l_knee_deg, r_knee_deg)
    asym_hip    = asymmetry_index(l_hip_deg,  r_hip_deg)
    asym_ankle  = asymmetry_index(l_ankle_deg, r_ankle_deg)
    asym_elbow  = asymmetry_index(l_elbow_deg, r_elbow_deg)
    asym_valgus = asymmetry_index(l_val, r_val)

    # ── Risk scores ──────────────────────────────────────────────────────────
    # Dynamic Valgus Score (0-10): combines knee valgus + hip drop + trunk lean
    dvs = min(10.0, (
        max(l_val, r_val) / 3.0 +
        abs(hip_drop) / 2.0 +
        trunk_lean / 10.0
    ))

    # LESS proxy: knee flexion < 30deg at near-ground-contact + valgus > 10deg
    near_ground  = min(l_foot_h, r_foot_h) < 0.05
    less_flag    = near_ground and (
        (l_knee_deg < 30 or r_knee_deg < 30) or
        (l_val > 10 or r_val > 10)
    )

    low_confidence = (l_knee_deg > 170 and r_knee_deg > 170)

    # ── Build base metrics dict ───────────────────────────────────────────────
    m = {
        # Lower body
        "l_knee_deg":    l_knee_deg,
        "r_knee_deg":    r_knee_deg,
        "l_hip_deg":     l_hip_deg,
        "r_hip_deg":     r_hip_deg,
        "l_ankle_deg":   l_ankle_deg,
        "r_ankle_deg":   r_ankle_deg,
        "l_valgus":      l_val,
        "r_valgus":      r_val,
        "kwr":           kwr,
        "hip_drop_cm":   hip_drop,
        "stride_len":    stride_len,
        "ground_contact": ground_contact,
        # Trunk / posture
        "trunk_lean":    trunk_lean,
        "forward_lean":  forward_lean,
        "head_tilt":     head_tilt,
        # Upper body
        "l_elbow_deg":   l_elbow_deg,
        "r_elbow_deg":   r_elbow_deg,
        "shoulder_rot":  shoulder_rot,
        # Asymmetry
        "asym_knee":     asym_knee,
        "asym_hip":      asym_hip,
        "asym_ankle":    asym_ankle,
        "asym_elbow":    asym_elbow,
        "asym_valgus":   asym_valgus,
        # Risk
        "dvs":           dvs,
        "less_flag":     less_flag,
        # CoM
        "com":           com,
        # Meta
        "low_confidence": low_confidence,
        # Temporal (defaults — overwritten below if buffer available)
        "knee_angular_vel_l": 0.0,
        "knee_angular_vel_r": 0.0,
        "hip_angular_vel_l":  0.0,
        "hip_angular_vel_r":  0.0,
        "com_sway_lateral":   0.0,
        "com_accel":          0.0,
        "cadence":            0.0,
        "rom_knee_l":         0.0,
        "rom_knee_r":         0.0,
        "rom_hip_l":          0.0,
        "rom_hip_r":          0.0,
    }

    # ── Temporal metrics (only when buffer is provided) ───────────────────────
    if buf is not None:
        buf.push(m)  # push BEFORE reading so we have at least 1 frame

        dt = 1.0 / fps

        # Angular velocities (deg/s)
        m["knee_angular_vel_l"] = abs(buf.delta("l_knee_deg")) / dt if len(buf.frames) >= 2 else 0.0
        m["knee_angular_vel_r"] = abs(buf.delta("r_knee_deg")) / dt if len(buf.frames) >= 2 else 0.0
        m["hip_angular_vel_l"]  = abs(buf.delta("l_hip_deg"))  / dt if len(buf.frames) >= 2 else 0.0
        m["hip_angular_vel_r"]  = abs(buf.delta("r_hip_deg"))  / dt if len(buf.frames) >= 2 else 0.0

        # CoM lateral sway (units match keypoint space)
        prev_com_vals = buf.last("com", 2)
        if len(prev_com_vals) == 2:
            com_delta = prev_com_vals[-1] - prev_com_vals[0]
            m["com_sway_lateral"] = float(abs(com_delta[0]))
            m["com_accel"]        = float(np.linalg.norm(com_delta) / dt)
        
        # Cadence: count ankle height zero-crossings (foot-strike events)
        ankle_h = buf.last("ground_contact", len(buf.frames))
        contacts = sum(1 for i in range(1, len(ankle_h)) if ankle_h[i] != ankle_h[i-1])
        m["cadence"] = float(contacts / (len(buf.frames) / fps)) if buf.frames else 0.0

        # Range of motion over buffer window
        m["rom_knee_l"] = buf.range_of_motion("l_knee_deg")
        m["rom_knee_r"] = buf.range_of_motion("r_knee_deg")
        m["rom_hip_l"]  = buf.range_of_motion("l_hip_deg")
        m["rom_hip_r"]  = buf.range_of_motion("r_hip_deg")

    return m

## 7. Rendering & video processing

`render_mesh_only` composites all player meshes depth-sorted onto a black canvas, draws the 2D skeleton, then overlays the five-panel biomechanics HUD.

`process_video_mesh` runs this per frame on a full video, maintaining a `TemporalBuffer` for rolling temporal metrics.

In [ ]:
def draw_metrics_overlay(frame, metrics, pos=(15, 30)):
    """
    Draw all biomechanical metrics as a structured HUD split into five panels:
    Posture | Lower Body | Upper Body | Risk | Temporal
    Colour codes: green = OK, orange = mild, red = concerning.
    """
    out  = frame.copy()
    font = cv2.FONT_HERSHEY_SIMPLEX
    fs   = 0.52
    fw   = 1
    lh   = 22  # line height
    pw   = 210 # panel width

    def col(val, warn, bad):
        if abs(val) >= bad:  return (0, 0, 220)
        if abs(val) >= warn: return (0, 140, 255)
        return (0, 200, 0)

    def put(img, text, x, y, color=(200, 200, 200), scale=None, weight=None):
        cv2.putText(img, text, (x, y), font,
                    scale or fs, color, weight or fw, cv2.LINE_AA)

    def panel_bg(img, x, y, w, h, alpha=0.45):
        overlay = img.copy()
        cv2.rectangle(overlay, (x, y), (x + w, y + h), (20, 20, 20), -1)
        cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0, img)

    if metrics.get("low_confidence"):
        panel_bg(out, pos[0]-5, pos[1]-20, 220, 35)
        put(out, "LOW CONFIDENCE", pos[0], pos[1], (0, 0, 220), 0.6, 2)
        return out

    panels = [
        ("POSTURE", [
            ("Trunk lean",    f"{metrics['trunk_lean']:.1f}deg",    col(metrics["trunk_lean"],    15, 30)),
            ("Fwd lean",      f"{metrics['forward_lean']:.1f}deg",  col(metrics["forward_lean"],  20, 40)),
            ("Head tilt",     f"{metrics['head_tilt']:.1f}deg",     col(metrics["head_tilt"],     10, 20)),
            ("Hip drop",      f"{metrics['hip_drop_cm']:.1f}cm",    col(abs(metrics["hip_drop_cm"]), 3, 6)),
            ("Shoulder rot",  f"{metrics['shoulder_rot']:.1f}deg",  col(metrics["shoulder_rot"],  15, 30)),
        ]),
        ("LOWER BODY", [
            ("L-Knee",        f"{metrics['l_knee_deg']:.1f}deg",    col(180-metrics["l_knee_deg"],  20, 40)),
            ("R-Knee",        f"{metrics['r_knee_deg']:.1f}deg",    col(180-metrics["r_knee_deg"],  20, 40)),
            ("L-Hip",         f"{metrics['l_hip_deg']:.1f}deg",     col(180-metrics["l_hip_deg"],   25, 50)),
            ("R-Hip",         f"{metrics['r_hip_deg']:.1f}deg",     col(180-metrics["r_hip_deg"],   25, 50)),
            ("L-Ankle",       f"{metrics['l_ankle_deg']:.1f}deg",   col(abs(metrics["l_ankle_deg"]-90), 15, 30)),
            ("R-Ankle",       f"{metrics['r_ankle_deg']:.1f}deg",   col(abs(metrics["r_ankle_deg"]-90), 15, 30)),
            ("L-Valgus",      f"{metrics['l_valgus']:.1f}deg",      col(metrics["l_valgus"],  10, 20)),
            ("R-Valgus",      f"{metrics['r_valgus']:.1f}deg",      col(metrics["r_valgus"],  10, 20)),
            ("KWR",           f"{metrics['kwr']:.2f}",              col(abs(metrics["kwr"]-1.0), 0.3, 0.6)),
            ("Stride",        f"{metrics['stride_len']:.2f}u",      (200, 200, 200)),
            ("Contact",       metrics["ground_contact"],             (200, 200, 200)),
        ]),
        ("UPPER BODY", [
            ("L-Elbow",       f"{metrics['l_elbow_deg']:.1f}deg",   (200, 200, 200)),
            ("R-Elbow",       f"{metrics['r_elbow_deg']:.1f}deg",   (200, 200, 200)),
        ]),
        ("RISK", [
            ("Dyn Valgus",    f"{metrics['dvs']:.1f}/10",           col(metrics["dvs"], 4, 7)),
            ("LESS flag",     "YES" if metrics["less_flag"] else "no",
                              (0, 0, 220) if metrics["less_flag"] else (0, 200, 0)),
            ("Asym Knee",     f"{metrics['asym_knee']:.1f}%",       col(metrics["asym_knee"],  15, 30)),
            ("Asym Hip",      f"{metrics['asym_hip']:.1f}%",        col(metrics["asym_hip"],   15, 30)),
            ("Asym Ankle",    f"{metrics['asym_ankle']:.1f}%",      col(metrics["asym_ankle"], 15, 30)),
            ("Asym Elbow",    f"{metrics['asym_elbow']:.1f}%",      col(metrics["asym_elbow"], 15, 30)),
            ("Asym Valgus",   f"{metrics['asym_valgus']:.1f}%",     col(metrics["asym_valgus"],15, 30)),
        ]),
        ("TEMPORAL", [
            ("Knee vel L",    f"{metrics['knee_angular_vel_l']:.0f}d/s", col(metrics["knee_angular_vel_l"], 200, 500)),
            ("Knee vel R",    f"{metrics['knee_angular_vel_r']:.0f}d/s", col(metrics["knee_angular_vel_r"], 200, 500)),
            ("Hip vel L",     f"{metrics['hip_angular_vel_l']:.0f}d/s",  col(metrics["hip_angular_vel_l"],  200, 500)),
            ("Hip vel R",     f"{metrics['hip_angular_vel_r']:.0f}d/s",  col(metrics["hip_angular_vel_r"],  200, 500)),
            ("CoM sway",      f"{metrics['com_sway_lateral']:.3f}u",     col(metrics["com_sway_lateral"], 0.05, 0.15)),
            ("CoM accel",     f"{metrics['com_accel']:.2f}u/s",          (200, 200, 200)),
            ("Cadence",       f"{metrics['cadence']:.1f}s/s",            (200, 200, 200)),
            ("ROM Knee L",    f"{metrics['rom_knee_l']:.1f}deg",         (200, 200, 200)),
            ("ROM Knee R",    f"{metrics['rom_knee_r']:.1f}deg",         (200, 200, 200)),
            ("ROM Hip L",     f"{metrics['rom_hip_l']:.1f}deg",          (200, 200, 200)),
            ("ROM Hip R",     f"{metrics['rom_hip_r']:.1f}deg",          (200, 200, 200)),
        ]),
    ]

    # Layout: stack panels left to right, wrap if needed
    img_h, img_w = out.shape[:2]
    col_x   = pos[0]
    col_y   = pos[1]
    max_col_h = img_h - 20

    for panel_title, rows in panels:
        panel_h = lh * (len(rows) + 1) + 10
        if col_y + panel_h > max_col_h:
            col_x += pw
            col_y  = pos[1]
        # Background
        panel_bg(out, col_x - 5, col_y - 18, pw - 5, panel_h)
        # Title
        put(out, panel_title, col_x, col_y, (255, 255, 255), 0.50, 2)
        col_y += lh
        for label, value, color in rows:
            put(out, f"{label}: {value}", col_x + 4, col_y, color)
            col_y += lh
        col_y += 6  # gap between panels

    return out


def render_mesh_only(outputs, faces, img_h, img_w, buf=None, fps=30.0):
    """
    Render mesh(es) onto a black canvas sorted by depth,
    then draw skeleton + full biomechanics HUD.

    Args:
        buf : TemporalBuffer instance (None for single-image use)
        fps : video fps for temporal metric calculation
    """
    all_depths = np.stack([p["pred_cam_t"] for p in outputs], axis=0)[:, 2]
    outputs_sorted = [outputs[i] for i in np.argsort(-all_depths)]

    all_pred_vertices, all_faces = [], []
    for pid, p in enumerate(outputs_sorted):
        all_pred_vertices.append(p["pred_vertices"] + p["pred_cam_t"])
        all_faces.append(faces + len(p["pred_vertices"]) * pid)
    all_pred_vertices = np.concatenate(all_pred_vertices, axis=0)
    all_faces         = np.concatenate(all_faces, axis=0)

    fake_cam_t = (
        np.max(all_pred_vertices[-2*18439:], axis=0) +
        np.min(all_pred_vertices[-2*18439:], axis=0)
    ) / 2
    all_pred_vertices -= fake_cam_t

    black_img = np.zeros((img_h, img_w, 3), dtype=np.uint8)
    renderer  = Renderer(focal_length=outputs_sorted[-1]["focal_length"], faces=all_faces)
    rend = (renderer(
        all_pred_vertices, fake_cam_t, black_img,
        mesh_base_color=LIGHT_BLUE, scene_bg_color=(0, 0, 0),
    ) * 255).astype(np.uint8)

    # Per-person skeleton + HUD
    for person_output in outputs_sorted:
        kp2d = person_output["pred_keypoints_2d"]
        kp2d = np.concatenate([kp2d, np.ones((kp2d.shape[0], 1))], axis=-1)
        rend = skeleton_visualizer.draw_skeleton(rend, kp2d)
        metrics = compute_biomechanics(person_output, buf=buf, fps=fps)
        rend    = draw_metrics_overlay(rend, metrics)

    return rend


def process_video_mesh(video_path, output_path="output_video.mp4"):
    """Process all detected players per frame with full biomechanics overlay."""
    cap    = cv2.VideoCapture(video_path)
    fps    = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total  = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
    buf    = TemporalBuffer(fps=fps, window=15)

    print(f"Input: {video_path}  |  {total} frames @ {fps:.1f}fps  |  {width}x{height}")
    try:
        for frame_idx in range(total if total > 0 else int(1e9)):
            ret, frame_bgr = cap.read()
            if not ret: break
            outputs = estimator.process_one_image(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
            rend    = render_mesh_only(outputs, estimator.faces, height, width, buf=buf, fps=fps) if outputs else np.zeros((height, width, 3), dtype=np.uint8)
            writer.write(rend)
            if (frame_idx + 1) % 10 == 0:
                print(f"  {frame_idx+1}/{total} frames")
    finally:
        cap.release()
        writer.release()
    print(f"Done -> {output_path}")
    return output_path

## 8. Player tracking

`_process_video_track_with_bbox` locks onto a single player via IoU on frame 0 and tracks them with ByteTrack. Each frame gets the full biomechanics HUD with a shared `TemporalBuffer`.

In [ ]:
def _process_video_track_with_bbox(video_path, seed_bbox, output_path="/kaggle/working/output_track.mp4"):
    """Track a single player seeded by bbox on frame 0, with full biomechanics HUD."""
    tracker = ByteTrack()
    cap     = cv2.VideoCapture(video_path)
    fps     = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total   = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height  = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    writer    = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (width, height))
    buf       = TemporalBuffer(fps=fps, window=15)
    target_id = None

    try:
        for frame_idx in range(total if total > 0 else int(1e9)):
            ret, frame_bgr = cap.read()
            if not ret: break

            frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            all_boxes = estimator.detector.run_human_detection(
                frame_bgr, det_cat_id=0, bbox_thr=0.5, nms_thr=0.3,
                default_to_full_image=False,
            )

            if len(all_boxes) == 0:
                writer.write(np.zeros((height, width, 3), dtype=np.uint8))
                continue

            dets   = np.hstack([all_boxes,
                                 np.ones((len(all_boxes), 1), dtype=np.float32),
                                 np.zeros((len(all_boxes), 1), dtype=np.float32)])
            tracks = tracker.update(dets, frame_bgr)

            if target_id is None and tracks is not None and len(tracks) and frame_idx == 0:
                sx1, sy1, sx2, sy2 = seed_bbox
                best_iou, best_id = 0.0, None
                for t in tracks:
                    tx1, ty1, tx2, ty2, tid = t[:5]
                    ix1, iy1 = max(sx1, tx1), max(sy1, ty1)
                    ix2, iy2 = min(sx2, tx2), min(sy2, ty2)
                    inter = max(0, ix2-ix1) * max(0, iy2-iy1)
                    if inter == 0: continue
                    union = (sx2-sx1)*(sy2-sy1) + (tx2-tx1)*(ty2-ty1) - inter
                    iou   = inter / (union + 1e-8)
                    if iou > best_iou:
                        best_iou, best_id = iou, int(tid)
                target_id = best_id
                print(f"Locked onto track ID {target_id} (IoU={best_iou:.2f})")

            target_bbox = None
            if tracks is not None:
                for t in tracks:
                    if int(t[4]) == target_id:
                        target_bbox = t[:4]
                        break

            if target_bbox is None:
                writer.write(np.zeros((height, width, 3), dtype=np.uint8))
                continue

            bbox_input = np.array(target_bbox, dtype=np.float32).reshape(1, 4)
            outputs    = estimator.process_one_image(frame_rgb, bboxes=bbox_input)

            if outputs:
                rend = render_mesh_only([outputs[0]], estimator.faces, height, width, buf=buf, fps=fps)
            else:
                rend = np.zeros((height, width, 3), dtype=np.uint8)

            writer.write(rend)
            if (frame_idx + 1) % 10 == 0:
                print(f"  {frame_idx+1}/{total} frames")
    finally:
        cap.release()
        writer.release()

## 9. .obj export

Exports reconstructed mesh(es) as Wavefront `.obj`. Vertices are Y/Z-flipped to match standard 3D viewer conventions.

In [ ]:
def save_obj(outputs, faces, output_path="output.obj"):
    """
    create a 3D object file of the player's mesh

    Args:
    outputs
    faces
    output_path

    Returns:
    3D object file
    """
    all_vertices = []
    for pid, person_output in enumerate(outputs):
        verts = person_output["pred_vertices"] + person_output["pred_cam_t"]
        all_vertices.append(verts)
    all_vertices = np.concatenate(all_vertices, axis=0)
    all_vertices[:, 1] *= -1  # flip Y
    all_vertices[:, 2] *= -1  # flip Z

    with open(output_path, "w") as f:
        for v in all_vertices:
            f.write(f"v {v[0]} {v[1]} {v[2]}\n")
        # .obj faces are 1-indexed
        for pid in range(len(outputs)):
            offset = len(person_output["pred_vertices"]) * pid
            for face in faces:
                f1, f2, f3 = face + offset + 1
                f.write(f"f {f1} {f2} {f3}\n")

    return output_path

## 10. Gradio UI

Four tabs:
1. **Single Image** — mesh reconstruction + all single-frame metrics as text
2. **Video — All Players** — full pipeline on all detected players per frame
3. **Video — Track Player** — preview frame 0 → select player by number → track with ByteTrack
4. **Export .obj** — download 3D mesh file

> **Player tracking steps:**
> 1. Upload video → **Preview** — players numbered on frame 0
> 2. Select player from the radio list
> 3. Click **Track**

In [ ]:
def metrics_to_text(metrics):
    """Format all metrics as readable text for the Gradio textbox."""
    sections = {
        "POSTURE": [
            ("Trunk lean",   f"{metrics['trunk_lean']:.1f}°"),
            ("Fwd lean",     f"{metrics['forward_lean']:.1f}°"),
            ("Head tilt",    f"{metrics['head_tilt']:.1f}°"),
            ("Hip drop",     f"{metrics['hip_drop_cm']:.1f} cm"),
            ("Shoulder rot", f"{metrics['shoulder_rot']:.1f}°"),
        ],
        "LOWER BODY": [
            ("L-Knee",    f"{metrics['l_knee_deg']:.1f}°"),
            ("R-Knee",    f"{metrics['r_knee_deg']:.1f}°"),
            ("L-Hip",     f"{metrics['l_hip_deg']:.1f}°"),
            ("R-Hip",     f"{metrics['r_hip_deg']:.1f}°"),
            ("L-Ankle",   f"{metrics['l_ankle_deg']:.1f}°"),
            ("R-Ankle",   f"{metrics['r_ankle_deg']:.1f}°"),
            ("L-Valgus",  f"{metrics['l_valgus']:.1f}°"),
            ("R-Valgus",  f"{metrics['r_valgus']:.1f}°"),
            ("KWR",       f"{metrics['kwr']:.2f}"),
            ("Stride",    f"{metrics['stride_len']:.2f} u"),
            ("Contact",   metrics["ground_contact"]),
        ],
        "UPPER BODY": [
            ("L-Elbow",   f"{metrics['l_elbow_deg']:.1f}°"),
            ("R-Elbow",   f"{metrics['r_elbow_deg']:.1f}°"),
        ],
        "RISK": [
            ("Dyn Valgus Score", f"{metrics['dvs']:.1f} / 10"),
            ("LESS flag",        "YES ⚠" if metrics["less_flag"] else "no"),
            ("Asym Knee",        f"{metrics['asym_knee']:.1f}%"),
            ("Asym Hip",         f"{metrics['asym_hip']:.1f}%"),
            ("Asym Ankle",       f"{metrics['asym_ankle']:.1f}%"),
            ("Asym Elbow",       f"{metrics['asym_elbow']:.1f}%"),
            ("Asym Valgus",      f"{metrics['asym_valgus']:.1f}%"),
        ],
    }
    lines = []
    for section, rows in sections.items():
        lines.append(f"── {section} ──")
        for label, value in rows:
            lines.append(f"  {label:<18}{value}")
        lines.append("")
    return "\n".join(lines)


import gradio as gr
import tempfile
import os

def run_process_image(img_array):
    """Process a single image and return mesh render + all metrics as text."""
    if img_array is None:
        return None, "No image provided."
    outputs = estimator.process_one_image(img_array)
    if not outputs:
        return None, "No person detected."
    h, w = img_array.shape[:2]
    rend    = render_mesh_only(outputs, estimator.faces, h, w, buf=None)
    metrics = compute_biomechanics(outputs[0], buf=None)
    return cv2.cvtColor(rend, cv2.COLOR_BGR2RGB), metrics_to_text(metrics)

def run_process_video_mesh(video_path):
    if video_path is None:
        return None, "No video provided."
    out_path = os.path.join(tempfile.mkdtemp(), "output_mesh.mp4")
    process_video_mesh(video_path, out_path)
    return out_path, f"Done → {out_path}"

def run_save_obj(img_array):
    if img_array is None:
        return None, "No image provided."
    outputs = estimator.process_one_image(img_array)
    if not outputs:
        return None, "No person detected."
    out_path = os.path.join(tempfile.mkdtemp(), "mesh.obj")
    save_obj(outputs, estimator.faces, out_path)
    return out_path, f"Saved → {out_path}"

# Shared state for player tracking
_preview_state = {"video_path": None, "detections": [], "frame": None}

def preview_first_frame(video_path):
    if video_path is None:
        return None, "No video provided.", gr.update(choices=[], visible=False)
    cap = cv2.VideoCapture(video_path)
    ret, frame_bgr = cap.read()
    cap.release()
    if not ret:
        return None, "Could not read video.", gr.update(choices=[], visible=False)
    boxes = estimator.detector.run_human_detection(
        frame_bgr, det_cat_id=0, bbox_thr=0.5, nms_thr=0.3,
        default_to_full_image=False,
    )
    preview = frame_bgr.copy()
    for idx, box in enumerate(boxes):
        x1, y1, x2, y2 = box[:4].astype(int)
        cv2.rectangle(preview, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(preview, f"#{idx}", (x1+5, y1+30),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 2, cv2.LINE_AA)
    _preview_state.update({"video_path": video_path, "detections": boxes, "frame": frame_bgr})
    n = len(boxes)
    if n == 0:
        return cv2.cvtColor(preview, cv2.COLOR_BGR2RGB), "No players detected.", gr.update(choices=[], visible=False)
    choices = [f"Player #{i}" for i in range(n)]
    return (cv2.cvtColor(preview, cv2.COLOR_BGR2RGB),
            f"{n} player(s) detected. Select one below, then click Track.",
            gr.update(choices=choices, value=choices[0], visible=True))

def run_track_selected(player_label):
    video_path = _preview_state["video_path"]
    boxes      = _preview_state["detections"]
    if video_path is None: return None, "Please preview a video first."
    if len(boxes) == 0:          return None, "No detections from preview step."
    if player_label is None: return None, "Please select a player."
    player_idx    = int(player_label.split("#")[1])
    selected_bbox = boxes[player_idx][:4]
    out_path      = "/kaggle/working/output_tracked.mp4"
    _process_video_track_with_bbox(video_path, selected_bbox, out_path)
    return out_path, f"Done → {out_path}"

with gr.Blocks(title="SAM 3D Body — CPU Offload") as demo:
    gr.Markdown("# SAM 3D Body — CPU Offload\n3D body reconstruction + biomechanics analysis for soccer.")

    with gr.Tab("Single Image"):
        gr.Markdown("Upload an image to reconstruct the 3D body mesh and compute all biomechanical metrics.")
        with gr.Row():
            with gr.Column():
                img_input  = gr.Image(label="Input Image", type="numpy")
                img_btn    = gr.Button("Run", variant="primary")
            with gr.Column():
                img_output  = gr.Image(label="Reconstruction")
                img_metrics = gr.Textbox(label="Metrics", lines=30)
        img_btn.click(run_process_image, inputs=img_input, outputs=[img_output, img_metrics])

    with gr.Tab("Video — All Players"):
        gr.Markdown("Process all detected players per frame with mesh + full biomechanics HUD.")
        with gr.Row():
            with gr.Column():
                vid_input  = gr.Video(label="Input Video")
                vid_btn    = gr.Button("Run", variant="primary")
            with gr.Column():
                vid_output = gr.Video(label="Output Video")
                vid_status = gr.Textbox(label="Status")
        vid_btn.click(run_process_video_mesh, inputs=vid_input, outputs=[vid_output, vid_status])

    with gr.Tab("Video — Track Player"):
        gr.Markdown(
            "**Step 1** — Upload a video and click **Preview** to detect players on frame 0.  \n"
            "**Step 2** — Select the player you want to track from the list.  \n"
            "**Step 3** — Click **Track** to process the full video with ByteTrack."
        )
        with gr.Row():
            with gr.Column():
                track_input   = gr.Video(label="Input Video")
                preview_btn   = gr.Button("Step 1 — Preview", variant="secondary")
                track_preview = gr.Image(label="Frame 0 — players numbered")
                preview_status = gr.Textbox(label="Status", interactive=False)
                player_radio  = gr.Radio(choices=[], label="Step 2 — Select player",
                                         visible=False, interactive=True)
                track_btn     = gr.Button("Step 3 — Track", variant="primary")
            with gr.Column():
                track_output = gr.Video(label="Output Video")
                track_status = gr.Textbox(label="Status", interactive=False)
        preview_btn.click(preview_first_frame, inputs=track_input,
                          outputs=[track_preview, preview_status, player_radio])
        track_btn.click(run_track_selected, inputs=player_radio,
                        outputs=[track_output, track_status])

    with gr.Tab("Export .obj"):
        gr.Markdown("Export the reconstructed 3D mesh as a Wavefront `.obj` file.")
        with gr.Row():
            with gr.Column():
                obj_input = gr.Image(label="Input Image", type="numpy")
                obj_btn   = gr.Button("Export", variant="primary")
            with gr.Column():
                obj_output = gr.File(label="Download .obj")
                obj_status = gr.Textbox(label="Status")
        obj_btn.click(run_save_obj, inputs=obj_input, outputs=[obj_output, obj_status])

In [ ]:
demo.launch(share=True, inline=False)